In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("purchases_product.csv")
df.head()

,purchase_id,customer_id,customer_name,purchase_date,product_ID,product_category,amount
0,1,CUST011,Harper Anderson,2025-05-21,ELEC-1001,Electronics,312.53
1,2,CUST003,Mia Brown,2025-12-13,SPRT-6001,Sports,928.04
2,3,CUST009,Amelia Martinez,2025-01-17,ELEC-1004,Electronics,45.49
3,4,CUST004,Liam Jones,2025-10-15,CLTH-2001,Clothing,255.36
4,5,CUST011,Harper Anderson,2025-04-23,SPRT-6004,Sports,566.19


In [ ]:
# =======================================================================
# Binary Search Implementation for Product ID
# =======================================================================
def binary_search_product(df, target_product_id, column="product_ID"):
    """
    Performs a binary search for a specific product ID in a pandas DataFrame.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        target_product_id (str): The product ID to search for (e.g., 'ELEC-1001').
        column (str): The column containing product IDs (default: 'product_ID').
        
    Returns:
        pd.Series or None: The matching row if found, otherwise None.
    """
    # 1. Binary search requires the data to be sorted
    sorted_df = df.sort_values(by=column).reset_index(drop=True)
    
    # 2. Initialize low and high pointers
    low = 0
    high = len(sorted_df) - 1
    
    # 3. Binary search loop
    while low <= high:
        mid = (low + high) // 2
        mid_val = sorted_df.iloc[mid][column]
        
        if mid_val == target_product_id:
            return sorted_df.iloc[mid]  # Match found
        elif mid_val < target_product_id:
            low = mid + 1               # Search right half
        else:
            high = mid - 1              # Search left half
            
    return None  # Not found


def binary_search_all_products(df, target_product_id, column="product_ID"):
    """
    Performs binary search and returns all rows matching the target product ID,
    handling cases where a product appears across multiple purchases.
    """
    sorted_df = df.sort_values(by=column).reset_index(drop=True)
    low, high = 0, len(sorted_df) - 1
    found_idx = -1
    
    while low <= high:
        mid = (low + high) // 2
        mid_val = sorted_df.iloc[mid][column]
        
        if mid_val == target_product_id:
            found_idx = mid
            break
        elif mid_val < target_product_id:
            low = mid + 1
        else:
            high = mid - 1
            
    if found_idx == -1:
        return pd.DataFrame()
        
    # Expand left and right to collect all duplicate product IDs
    start = found_idx
    while start > 0 and sorted_df.iloc[start - 1][column] == target_product_id:
        start -= 1
    end = found_idx
    while end < len(sorted_df) - 1 and sorted_df.iloc[end + 1][column] == target_product_id:
        end += 1
        
    return sorted_df.iloc[start:end + 1]


# --- Demonstration ---
target_id = "ELEC-1001"
print(f"=== Single Match Binary Search for '{target_id}' ===")
match = binary_search_product(df, target_id)
if match is not None:
    print(match)
else:
    print("Product ID not found.")

print(f"\n=== All Transactions for '{target_id}' ===")
all_matches = binary_search_all_products(df, target_id)
print(all_matches)


In [ ]:
# =======================================================================
# 1. Sequential Search Function on Unindexed Data
# =======================================================================
def sequential_search_product(df, target_product_id, column="product_ID"):
    """
    Performs a sequential (linear) search for a specific product ID
    in an unsorted, unindexed DataFrame.
    
    Parameters:
        df (pd.DataFrame): The unindexed DataFrame.
        target_product_id (str): The product ID to search for.
        column (str): The column containing product IDs (default: 'product_ID').
        
    Returns:
        pd.Series or None: The matching row if found, otherwise None.
    """
    for i in range(len(df)):
        if df.iloc[i][column] == target_product_id:
            return df.iloc[i]
    return None


# =======================================================================
# 2. Performance Simulation & Comparison Function
# =======================================================================
import time

def simulate_search_performance(df, targets, iterations=500, column="product_ID"):
    """
    Simulates and compares performance between Sequential Search (unindexed)
    and Binary Search (sorted index).
    
    Measures:
        - Number of comparisons made
        - Average execution time (microseconds per search)
        - Speedup ratio
    """
    sorted_df = df.sort_values(by=column).reset_index(drop=True)
    results = []
    
    for target in targets:
        # Count comparisons for Sequential Search
        seq_comps = 0
        for i in range(len(df)):
            seq_comps += 1
            if df.iloc[i][column] == target:
                break
                
        # Count comparisons for Binary Search
        bin_comps = 0
        low, high = 0, len(sorted_df) - 1
        while low <= high:
            bin_comps += 1
            mid = (low + high) // 2
            val = sorted_df.iloc[mid][column]
            if val == target:
                break
            elif val < target:
                low = mid + 1
            else:
                high = mid - 1
                
        # Benchmark timing for Sequential Search
        t0 = time.perf_counter()
        for _ in range(iterations):
            sequential_search_product(df, target, column)
        seq_time_us = ((time.perf_counter() - t0) / iterations) * 1_000_000
        
        # Benchmark timing for Binary Search
        t0 = time.perf_counter()
        for _ in range(iterations):
            binary_search_product(sorted_df, target, column)
        bin_time_us = ((time.perf_counter() - t0) / iterations) * 1_000_000
        
        ratio = seq_time_us / bin_time_us if bin_time_us > 0 else float('inf')
        
        results.append({
            "Target ID": target,
            "Scenario": "Best Case (Row 0)" if seq_comps == 1 else ("Worst Case (Not Found)" if target not in df[column].values else "Average Case"),
            "Seq Comps": seq_comps,
            "Bin Comps": bin_comps,
            "Seq Time (µs)": round(seq_time_us, 2),
            "Bin Time (µs)": round(bin_time_us, 2),
            "Speedup (Seq/Bin)": f"{ratio:.2f}x"
        })
        
    comparison_df = pd.DataFrame(results)
    return comparison_df


# --- Run Simulation across Key Test Cases ---
# 1. Best Case for Sequential (first row: ELEC-1001)
# 2. Typical Case (HOME-5004)
# 3. Worst Case for Sequential (item not found: NON-EXISTENT)
test_cases = ["ELEC-1001", "HOME-5004", "NON-EXISTENT"]
comparison_results = simulate_search_performance(df, test_cases, iterations=500)
print("=== Performance Comparison: Sequential vs Binary Search ===")
display(comparison_results)
